[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/gouravkhanijoe13/agentic-ai-lab/blob/main/Lesson_55_Ship_It.ipynb)

# Lesson 55 — Phase 5 Capstone: Ship It! 🚀
**AI Engineering Curriculum · Phase 5 · Lesson 5 of 5**

---

## Phase 5 Roadmap — COMPLETE

| Lesson | Topic | Status |
|--------|-------|--------|
| L51 | OSS Architecture & Scaffold | ✅ Done |
| L52 | Advanced Retrieval: Beyond Naive RAG | ✅ Done |
| L53 | Production Deployment | ✅ Done |
| L54 | Developer Experience (DevEx) | ✅ Done |
| **L55** | **Ship It: Integration Test → GitHub Release → PyPI → Badges** | **← You are here** |

---

## What You'll Build Today

This is the capstone of **Phase 5** — and the capstone of the **entire curriculum**. You will:

1. **Write an integration smoke test** that exercises the real pipeline end-to-end
2. **Cut a GitHub release** with automated CHANGELOG extraction
3. **Publish to PyPI** via Twine (with a TestPyPI dry-run first)
4. **Add README badges** — CI status, PyPI version, coverage, license
5. **Wire a full CI/CD GitHub Actions pipeline** (test → build → publish on tag)
6. **Retrospect** on everything you've built across 5 tracks + 5 phases

After this lesson, `pip install auto-researcher` will install **your code** from PyPI.

> **Key insight:** Shipping is a skill. Most projects die in the gap between "it works locally" and "it's on PyPI." This lesson closes that gap.

In [ ]:
# ── SETUP ─────────────────────────────────────────────────────────────────────
# Install all deps needed for this lesson
!pip install anthropic pydantic pydantic-settings build twine pytest pytest-asyncio \
             httpx rich tabulate nest_asyncio -q

# Load API key from Colab Secrets (Key name: ANTHROPIC_API_KEY)
import os
try:
    from google.colab import userdata
    os.environ["ANTHROPIC_API_KEY"] = userdata.get("ANTHROPIC_API_KEY")
    print("✅ API key loaded from Colab Secrets")
except Exception:
    print("⚠️  Not in Colab or secret missing — set ANTHROPIC_API_KEY manually")
    # os.environ["ANTHROPIC_API_KEY"] = "sk-ant-..."

import asyncio, json, subprocess, sys, textwrap, time
from pathlib import Path
from dataclasses import dataclass, field
from typing import Optional
import nest_asyncio
nest_asyncio.apply()

ROOT = Path("/content/auto_researcher_v2")
ROOT.mkdir(parents=True, exist_ok=True)
print(f"📁 Project root: {ROOT}")

---
## Part 1: Integration Smoke Test

### Unit tests vs Integration smoke tests

You already have unit tests (L54). They test components in isolation with mocks. But they can't catch **wiring bugs** — the kind that only surface when the whole system runs together.

An **integration smoke test** is a minimal end-to-end exercise:
- Uses the *real* code paths (no mocks)
- Hits the *real* API (costs a few cents)
- Asks a simple question to verify the pipeline returns a coherent answer
- Runs in < 60 seconds
- **Must pass before any release is cut**

```
smoke test anatomy
──────────────────────────────────────────────────────────
1. Import the public package interface
2. Build with minimal config (no flags, just API key)
3. Run one query
4. Assert structural correctness (not semantic quality)
5. Assert cost is within budget ($0.05 max)
6. Report pass/fail + timing + cost
──────────────────────────────────────────────────────────
```

**Rule:** Never release a version that fails the smoke test.

In [ ]:
# ── PART 1: Integration Smoke Test ────────────────────────────────────────────
import anthropic
from pydantic import BaseModel

# ─── Minimal pipeline (mirrors auto_researcher_v2 architecture) ───────────────
class ResearchResult(BaseModel):
    query: str
    answer: str
    sub_questions: list[str]
    cost_usd: float
    latency_s: float

async def research(query: str, max_cost_usd: float = 0.05) -> ResearchResult:
    """Minimal production-equivalent pipeline for smoke testing."""
    client = anthropic.AsyncAnthropic()
    t0 = time.perf_counter()
    total_cost = 0.0

    # ── Step 1: Plan (Sonnet – decompose into sub-questions) ──────────────────
    plan_resp = await client.messages.create(
        model="claude-sonnet-4-5",
        max_tokens=300,
        messages=[{"role": "user", "content": (
            f"Break this research question into 3 focused sub-questions.\n"
            f"Return a JSON array of strings only, no prose.\n\nQuestion: {query}"
        )}]
    )
    plan_cost = (plan_resp.usage.input_tokens * 3 + plan_resp.usage.output_tokens * 15) / 1_000_000
    total_cost += plan_cost

    raw_plan = plan_resp.content[0].text.strip()
    # Tolerate markdown code fences
    raw_plan = raw_plan.removeprefix("```json").removeprefix("```").removesuffix("```").strip()
    sub_questions = json.loads(raw_plan)
    assert isinstance(sub_questions, list) and len(sub_questions) >= 1

    if total_cost > max_cost_usd:
        raise RuntimeError(f"Budget exceeded after planning: ${total_cost:.4f}")

    # ── Step 2: Search (Haiku – answer each sub-question cheaply) ─────────────
    search_results = []
    for sq in sub_questions[:3]:   # cap at 3 even if model returns more
        sr = await client.messages.create(
            model="claude-haiku-4-5",
            max_tokens=200,
            messages=[{"role": "user", "content": f"Answer concisely (2-3 sentences): {sq}"}]
        )
        search_results.append(sr.content[0].text.strip())
        search_cost = (sr.usage.input_tokens * 0.8 + sr.usage.output_tokens * 4) / 1_000_000
        total_cost += search_cost

    if total_cost > max_cost_usd:
        raise RuntimeError(f"Budget exceeded after search: ${total_cost:.4f}")

    # ── Step 3: Draft (Sonnet – synthesize final answer) ──────────────────────
    findings = "\n".join(f"[{i+1}] {r}" for i, r in enumerate(search_results))
    draft_resp = await client.messages.create(
        model="claude-sonnet-4-5",
        max_tokens=400,
        messages=[{"role": "user", "content": (
            f"Synthesize these research findings into a coherent 2-paragraph answer.\n\n"
            f"Original question: {query}\n\nFindings:\n{findings}"
        )}]
    )
    draft_cost = (draft_resp.usage.input_tokens * 3 + draft_resp.usage.output_tokens * 15) / 1_000_000
    total_cost += draft_cost
    answer = draft_resp.content[0].text.strip()

    latency = time.perf_counter() - t0
    return ResearchResult(
        query=query,
        answer=answer,
        sub_questions=sub_questions,
        cost_usd=total_cost,
        latency_s=latency,
    )


# ─── Smoke test runner ────────────────────────────────────────────────────────
@dataclass
class SmokeResult:
    name: str
    passed: bool
    error: Optional[str] = None
    details: dict = field(default_factory=dict)

async def run_smoke_tests() -> list[SmokeResult]:
    results: list[SmokeResult] = []

    # ── Test 1: Happy path ────────────────────────────────────────────────────
    try:
        r = await research("What is retrieval-augmented generation?")
        assert len(r.answer) > 50,     "Answer too short"
        assert len(r.sub_questions) >= 1, "No sub-questions generated"
        assert r.cost_usd < 0.05,      f"Cost ${r.cost_usd:.4f} exceeds $0.05 budget"
        assert r.latency_s < 60,       f"Latency {r.latency_s:.1f}s exceeds 60s timeout"
        results.append(SmokeResult(
            "happy_path", True,
            details={"cost": f"${r.cost_usd:.4f}", "latency": f"{r.latency_s:.1f}s",
                     "answer_len": len(r.answer), "sub_q_count": len(r.sub_questions)}
        ))
    except Exception as e:
        results.append(SmokeResult("happy_path", False, str(e)))

    # ── Test 2: Budget enforcement ────────────────────────────────────────────
    try:
        # Set $0.000001 budget — should raise immediately after planning
        await research("What is an AI agent?", max_cost_usd=0.000001)
        results.append(SmokeResult("budget_enforcement", False, "Should have raised RuntimeError"))
    except RuntimeError as e:
        if "Budget exceeded" in str(e):
            results.append(SmokeResult("budget_enforcement", True, details={"raised": str(e)[:60]}))
        else:
            results.append(SmokeResult("budget_enforcement", False, str(e)))
    except Exception as e:
        results.append(SmokeResult("budget_enforcement", False, str(e)))

    # ── Test 3: Result schema completeness ────────────────────────────────────
    try:
        r = await research("Name one benefit of transformer architecture")
        fields_ok = all([
            isinstance(r.query, str) and r.query,
            isinstance(r.answer, str) and r.answer,
            isinstance(r.sub_questions, list),
            isinstance(r.cost_usd, float) and r.cost_usd > 0,
            isinstance(r.latency_s, float) and r.latency_s > 0,
        ])
        assert fields_ok, "One or more result fields invalid"
        results.append(SmokeResult("result_schema", True,
                                   details={"cost": f"${r.cost_usd:.4f}", "latency": f"{r.latency_s:.1f}s"}))
    except Exception as e:
        results.append(SmokeResult("result_schema", False, str(e)))

    return results


# ─── Run ──────────────────────────────────────────────────────────────────────
print("Running integration smoke tests...\n")
smoke_results = asyncio.run(run_smoke_tests())

all_passed = all(r.passed for r in smoke_results)
for r in smoke_results:
    icon = "✅" if r.passed else "❌"
    detail_str = "  " + str(r.details) if r.details else ""
    err_str    = f"  ERROR: {r.error}" if r.error else ""
    print(f"{icon} {r.name}{detail_str}{err_str}")

print()
if all_passed:
    print("🟢 ALL SMOKE TESTS PASSED — safe to release")
else:
    print("🔴 SMOKE TESTS FAILED — do NOT release until fixed")

# 💡 EXPERIMENT: Add a 4th test that checks the answer mentions 'retrieval' or 'generation'
# for semantic correctness — a weak but cheap sanity check.

---
## Part 2: GitHub Release with `gh` CLI

A GitHub release is more than a git tag. It:
- Creates a **tagged snapshot** of the codebase at version `v1.0.0`
- Publishes **release notes** (extracted from CHANGELOG.md)
- Triggers any **release-gated CI** (the PyPI publish step)
- Gives users a **stable URL** to reference

### Release workflow

```
smoke tests pass
      │
      ▼
bump version in pyproject.toml
      │
      ▼
update CHANGELOG.md  [Unreleased] → [1.0.0] - 2026-06-27
      │
      ▼
git commit -m "chore: release v1.0.0"
      │
      ▼
git tag v1.0.0
      │
      ▼
git push origin main --tags
      │
      ▼
gh release create v1.0.0 --notes-from-tag  ← or extract CHANGELOG section
      │
      ▼
GitHub Actions: on: push: tags: ['v*']  → run CI → publish to PyPI
```

The `gh` CLI automates the release creation step. Here's the full script.

In [ ]:
# ── PART 2: GitHub Release Script ─────────────────────────────────────────────
# This generates the release automation script you'd run locally (not in Colab).
# Colab doesn't have git + gh installed pointing to YOUR repo,
# so we generate the script and explain each step.

RELEASE_SCRIPT = '''
#!/usr/bin/env bash
# release.sh — Run from the root of auto_researcher_v2 to cut a release
set -euo pipefail

VERSION=${1:?"Usage: ./release.sh 1.0.0"}
TAG="v${VERSION}"
TODAY=$(date +%Y-%m-%d)

echo "━━━ Pre-release checks ━━━"

# 1. Require clean working tree
if [[ -n $(git status --porcelain) ]]; then
  echo "❌ Uncommitted changes detected. Commit or stash before releasing."
  exit 1
fi

# 2. Run tests
echo "Running tests..."
pytest tests/ -q --tb=short

# 3. Run smoke test (requires ANTHROPIC_API_KEY in env)
echo "Running integration smoke test..."
python -c "
import asyncio
from tests.smoke import run_smoke_tests
results = asyncio.run(run_smoke_tests())
failed = [r for r in results if not r.passed]
if failed:
    print('SMOKE FAILED:', [r.name for r in failed])
    exit(1)
print('Smoke OK')
"

echo "━━━ Updating version ━━━"

# 4. Bump version in pyproject.toml
sed -i "s/^version = .*/version = \"${VERSION}\"/" pyproject.toml

# 5. Rotate CHANGELOG [Unreleased] -> [VERSION]
sed -i "s/## \[Unreleased\]/## [${VERSION}] - ${TODAY}/" CHANGELOG.md

# 6. Extract release notes from CHANGELOG for this version
NOTES=$(awk "/## \\[${VERSION}\\]/,/## \\[/" CHANGELOG.md | head -n -1)

echo "━━━ Committing and tagging ━━━"

# 7. Commit + tag
git add pyproject.toml CHANGELOG.md
git commit -m "chore: release ${TAG}"
git tag -a "${TAG}" -m "Release ${TAG}"
git push origin main --tags

echo "━━━ Creating GitHub release ━━━"

# 8. Create GitHub release (this triggers the Actions publish workflow)
gh release create "${TAG}" \\
  --title "${TAG}" \\
  --notes "${NOTES}" \\
  --verify-tag

echo "✅ Released ${TAG} — GitHub Actions will publish to PyPI."
'''

script_path = ROOT / "release.sh"
script_path.write_text(RELEASE_SCRIPT.lstrip())
print("Generated release.sh")

# ─── Explain each step ───────────────────────────────────────────────────────
steps = [
    ("Clean tree check",   "Prevents releasing with uncommitted debug code"),
    ("pytest",             "All unit tests pass before anything goes out"),
    ("smoke test",         "Real end-to-end pipeline works with live API"),
    ("Bump pyproject.toml","version = '1.0.0' becomes the source of truth"),
    ("Rotate CHANGELOG",   "[Unreleased] becomes [1.0.0] - 2026-06-27"),
    ("Extract notes",      "Populates gh release description automatically"),
    ("git tag + push",     "Triggers tag-gated CI (the publish workflow)"),
    ("gh release create",  "Creates GitHub Release page with extracted notes"),
]

print("\nrelease.sh steps:")
print("-" * 62)
for step, reason in steps:
    print(f"  {step:<22}  {reason}")

print("\n💡 EXPERIMENT: Run `gh release view v1.0.0` after the script")
print("   to see the release page. Add --json to get machine-readable output.")

---
## Part 3: Publishing to PyPI with Twine

PyPI is Python's package index. Publishing to it means anyone can `pip install auto-researcher`.

### The packaging mental model

```
pyproject.toml          ← metadata: name, version, deps, entry points
      │
      ▼
python -m build         ← creates dist/
      │                     auto_researcher-1.0.0.tar.gz   (source dist)
      │                     auto_researcher-1.0.0-py3-none-any.whl  (wheel)
      ▼
twine check dist/*      ← validate metadata before uploading
      │
      ▼
twine upload --repository testpypi dist/*   ← dry-run on test.pypi.org
      │
      ▼
pip install --index-url https://test.pypi.org/simple/ auto-researcher
      │
      ▼ (if all good)
twine upload dist/*     ← real pypi.org
```

### Trusted publishing (modern, token-free)

Instead of storing a PyPI token in GitHub Secrets, use **OIDC Trusted Publishing** — PyPI trusts GitHub Actions directly. No long-lived secrets. Set it up once in your PyPI project settings.

In [ ]:
# ── PART 3: PyPI Publishing Pipeline ─────────────────────────────────────────

# ─── 3a. Minimal production-ready pyproject.toml ─────────────────────────────
PYPROJECT = '''
[build-system]
requires = ["setuptools>=68", "wheel"]
build-backend = "setuptools.backends.legacy:build"

[project]
name = "auto-researcher"
version = "1.0.0"
description = "A production-grade AI research assistant built with the Anthropic SDK."
readme = "README.md"
license = { text = "MIT" }
requires-python = ">=3.10"
authors = [
  { name = "Gourav Khanijoe", email = "gouravkhanijoe@gmail.com" },
]
keywords = ["ai", "agents", "llm", "rag", "research", "anthropic"]
classifiers = [
  "Development Status :: 4 - Beta",
  "Intended Audience :: Developers",
  "License :: OSI Approved :: MIT License",
  "Programming Language :: Python :: 3",
  "Programming Language :: Python :: 3.10",
  "Programming Language :: Python :: 3.11",
  "Programming Language :: Python :: 3.12",
  "Topic :: Scientific/Engineering :: Artificial Intelligence",
]
dependencies = [
  "anthropic>=0.40",
  "pydantic>=2.0",
  "pydantic-settings>=2.0",
]

[project.optional-dependencies]
multimodal = ["openai>=1.0", "pdfplumber>=0.10", "fpdf2>=2.7", "openai-whisper"]
serving    = ["fastapi>=0.110", "uvicorn[standard]>=0.29"]
cli        = ["typer>=0.12"]
all        = ["auto-researcher[multimodal,serving,cli]"]

[project.scripts]
auto-researcher = "auto_researcher.cli:app"

[project.urls]
Homepage      = "https://github.com/gouravkhanijoe/auto-researcher"
Documentation = "https://gouravkhanijoe.github.io/auto-researcher"
Repository    = "https://github.com/gouravkhanijoe/auto-researcher"
"Bug Tracker" = "https://github.com/gouravkhanijoe/auto-researcher/issues"

[tool.setuptools.packages.find]
where = ["."]   # finds auto_researcher/ package

[tool.ruff]
line-length = 100
target-version = "py310"

[tool.mypy]
python_version = "3.10"
strict = true
ignore_missing_imports = true

[tool.pytest.ini_options]
asyncio_mode = "auto"
testpaths = ["tests"]
markers = ["integration: marks tests that hit real APIs (deselect with -m 'not integration')"]
'''

(ROOT / "pyproject.toml").write_text(PYPROJECT.lstrip())
print("✅ Written pyproject.toml")

# ─── 3b. Build the package ────────────────────────────────────────────────────
print("\nBuilding dist/ ...")

# Create a minimal package for the build demo
pkg_dir = ROOT / "auto_researcher"
pkg_dir.mkdir(exist_ok=True)
(pkg_dir / "__init__.py").write_text(
    '__version__ = "1.0.0"\n__all__ = ["__version__"]\n'
)
(ROOT / "README.md").write_text(
    "# auto-researcher\n\nA production-grade AI research assistant.\n"
)

result = subprocess.run(
    [sys.executable, "-m", "build", "--outdir", str(ROOT / "dist")],
    cwd=str(ROOT), capture_output=True, text=True
)
if result.returncode == 0:
    dist_files = list((ROOT / "dist").glob("*"))
    print("✅ Build succeeded:")
    for f in dist_files:
        print(f"   {f.name}  ({f.stat().st_size:,} bytes)")
else:
    print("Build stderr:", result.stderr[-500:])

# ─── 3c. Validate with twine check ───────────────────────────────────────────
print("\nRunning twine check ...")
check = subprocess.run(
    [sys.executable, "-m", "twine", "check", str(ROOT / "dist" / "*")],
    cwd=str(ROOT), capture_output=True, text=True, shell=False
)
# twine check needs glob expansion — call via shell
check2 = subprocess.run(
    f"{sys.executable} -m twine check {ROOT}/dist/*",
    shell=True, capture_output=True, text=True
)
output = (check2.stdout + check2.stderr).strip()
print(output if output else "(twine check output empty — dist/ may be empty)")

# ─── 3d. Upload commands (don't actually upload from Colab) ──────────────────
print("\n" + "─" * 60)
print("To publish (run locally after `python -m build`):")
print()
print("  # Dry-run to TestPyPI first:")
print("  twine upload --repository testpypi dist/*")
print()
print("  # Install from TestPyPI to verify:")
print("  pip install --index-url https://test.pypi.org/simple/ auto-researcher")
print()
print("  # Real upload (only if TestPyPI install works):")
print("  twine upload dist/*")
print()
print("  # Or use OIDC Trusted Publishing from GitHub Actions (no token needed)")
print("  # — see Part 5 for the workflow file.")

# 💡 EXPERIMENT: Replace twine with 'uv publish' for faster uploads (uv is twine's spiritual successor)

---
## Part 4: README Badges

Badges are the first thing a developer looks at on your repo page. They signal:
- **Is CI passing?** (GitHub Actions badge)
- **What version is on PyPI?** (PyPI version badge)
- **Is it tested?** (coverage badge)
- **Is it safe to use?** (license badge)
- **Does it support my Python?** (Python version badge)

All badges come from [shields.io](https://shields.io) — a free service that renders SVG badges on-the-fly from public data sources.

### Badge anatomy

```
https://img.shields.io/github/actions/workflow/status/{owner}/{repo}/{workflow}.yml
                                                     ↑         ↑       ↑
                                               your GitHub user  repo  .github/workflows/ci.yml
```

In [ ]:
# ── PART 4: README Badges ─────────────────────────────────────────────────────
GITHUB_USER = "gouravkhanijoe"
GITHUB_REPO = "auto-researcher"
PYPI_NAME   = "auto-researcher"

# ─── Badge definitions ────────────────────────────────────────────────────────
badges = [
    (
        "CI",
        f"https://github.com/{GITHUB_USER}/{GITHUB_REPO}/actions/workflows/ci.yml/badge.svg",
        f"https://github.com/{GITHUB_USER}/{GITHUB_REPO}/actions/workflows/ci.yml",
    ),
    (
        "PyPI version",
        f"https://img.shields.io/pypi/v/{PYPI_NAME}.svg",
        f"https://pypi.org/project/{PYPI_NAME}/",
    ),
    (
        "Python",
        f"https://img.shields.io/pypi/pyversions/{PYPI_NAME}.svg",
        f"https://pypi.org/project/{PYPI_NAME}/",
    ),
    (
        "License",
        f"https://img.shields.io/github/license/{GITHUB_USER}/{GITHUB_REPO}.svg",
        f"https://github.com/{GITHUB_USER}/{GITHUB_REPO}/blob/main/LICENSE",
    ),
    (
        "Coverage",
        f"https://img.shields.io/codecov/c/github/{GITHUB_USER}/{GITHUB_REPO}.svg",
        f"https://codecov.io/gh/{GITHUB_USER}/{GITHUB_REPO}",
    ),
    (
        "Downloads",
        f"https://img.shields.io/pypi/dm/{PYPI_NAME}.svg",
        f"https://pypi.org/project/{PYPI_NAME}/",
    ),
]

# ─── Generate badge markdown ──────────────────────────────────────────────────
badge_md_lines = []
for name, img_url, link_url in badges:
    badge_md_lines.append(f"[![{name}]({img_url})]({link_url})")

badge_block = " ".join(badge_md_lines)

# ─── Full README.md ──────────────────────────────────────────────────────────
README = f"""# auto-researcher

{badge_block}

A production-grade AI research assistant powered by the Anthropic SDK.
Decomposes complex questions into sub-questions, searches in parallel,
and synthesizes a cited answer — all within a configurable cost budget.

## Installation

```bash
pip install auto-researcher

# With optional extras:
pip install auto-researcher[multimodal]   # voice + image + PDF
pip install auto-researcher[serving]      # FastAPI server
pip install auto-researcher[all]          # everything
```

## Quick Start

```python
import asyncio
from auto_researcher import AutoResearcher

ar = AutoResearcher()  # reads ANTHROPIC_API_KEY from env
result = asyncio.run(ar.research("What is retrieval-augmented generation?"))
print(result.answer)
```

## CLI

```bash
auto-researcher research "What are the trade-offs of fine-tuning vs RAG?"
auto-researcher serve   # Starts FastAPI server on :8000
```

## Architecture

```
User Query
    │
    ▼
Plan (Sonnet)  ──► Sub-questions [1, 2, 3]
    │                      │
    │           parallel asyncio.gather
    │                      │
    ▼                      ▼
Draft (Sonnet)  ◄── Search results (Haiku × N)
    │
    ▼
Critique → Revise if score < 0.65
    │
    ▼
ResearchResult (answer + citations + cost)
```

## Contributing

See [CONTRIBUTING.md](CONTRIBUTING.md). We follow [Conventional Commits](https://conventionalcommits.org).

## License

MIT
"""

(ROOT / "README.md").write_text(README)
print("✅ Generated README.md with badges")

# ─── Print badge block for inspection ────────────────────────────────────────
print("\nBadge markdown:")
print("-" * 60)
for line in badge_md_lines:
    print(" ", line)

print("\nBadge notes:")
notes = [
    ("CI badge",       "auto-updates on each Actions run"),
    ("PyPI version",   "auto-updates when you twine upload"),
    ("Coverage",       "requires codecov.io integration in CI"),
    ("Downloads",      "shows real installs — motivating to watch grow!"),
]
for name, note in notes:
    print(f"  {name:<18} {note}")

# 💡 EXPERIMENT: Add a 'code style' badge pointing to ruff and a
# 'docs' badge pointing to your mkdocs-material site on GitHub Pages.

---
## Part 5: Full CI/CD with GitHub Actions

Two workflows power the complete release pipeline:

| Workflow | Trigger | What it does |
|----------|---------|-------------|
| `ci.yml` | Every push / PR | Lint → test (matrix 3.10/3.11/3.12) → coverage → docs build |
| `release.yml` | Push tag `v*` | Build → twine check → publish to PyPI via OIDC |

### OIDC Trusted Publishing (no tokens)

With Trusted Publishing, GitHub Actions proves its identity to PyPI via a short-lived OpenID Connect token — no long-lived PyPI API token to rotate or leak. Setup:

1. Go to pypi.org → Your project → Publishing
2. Add Trusted Publisher: `owner=gouravkhanijoe`, `repo=auto-researcher`, `workflow=release.yml`
3. Done — no secrets needed in GitHub.

In [ ]:
# ── PART 5: GitHub Actions Workflows ─────────────────────────────────────────
gha_dir = ROOT / ".github" / "workflows"
gha_dir.mkdir(parents=True, exist_ok=True)

# ─── ci.yml ──────────────────────────────────────────────────────────────────
CI_YML = '''
name: CI

on:
  push:
    branches: [main]
  pull_request:
    branches: [main]

jobs:
  lint:
    runs-on: ubuntu-latest
    steps:
      - uses: actions/checkout@v4
      - uses: astral-sh/ruff-action@v1
        with:
          args: "check . --output-format=github"

  test:
    needs: lint
    runs-on: ubuntu-latest
    strategy:
      fail-fast: false
      matrix:
        python-version: ["3.10", "3.11", "3.12"]
    steps:
      - uses: actions/checkout@v4

      - uses: actions/setup-python@v5
        with:
          python-version: ${{ matrix.python-version }}
          cache: pip

      - name: Install deps
        run: pip install -e ".[all]" pytest pytest-asyncio pytest-cov

      - name: Run unit tests (no API calls)
        run: pytest tests/ -m "not integration" --cov=auto_researcher --cov-report=xml -q

      - name: Upload coverage
        uses: codecov/codecov-action@v4
        with:
          token: ${{ secrets.CODECOV_TOKEN }}
          file: coverage.xml

  smoke:
    # Only run smoke test on main branch pushes (not PRs — saves API costs)
    needs: test
    if: github.ref == 'refs/heads/main' && github.event_name == 'push'
    runs-on: ubuntu-latest
    steps:
      - uses: actions/checkout@v4
      - uses: actions/setup-python@v5
        with:
          python-version: "3.11"
          cache: pip
      - name: Install
        run: pip install -e .
      - name: Integration smoke test
        env:
          ANTHROPIC_API_KEY: ${{ secrets.ANTHROPIC_API_KEY }}
        run: python -m pytest tests/smoke_test.py -m integration -v

  docs:
    needs: lint
    runs-on: ubuntu-latest
    steps:
      - uses: actions/checkout@v4
      - uses: actions/setup-python@v5
        with:
          python-version: "3.11"
          cache: pip
      - run: pip install mkdocs-material mkdocstrings[python]
      - run: mkdocs build --strict   # fails if any doc reference is broken
'''

(gha_dir / "ci.yml").write_text(CI_YML.lstrip())
print("✅ Written .github/workflows/ci.yml")

# ─── release.yml ─────────────────────────────────────────────────────────────
RELEASE_YML = '''
name: Publish to PyPI

on:
  push:
    tags:
      - "v*"   # triggers on v1.0.0, v1.1.0, etc.

jobs:
  publish:
    runs-on: ubuntu-latest

    # Required for OIDC trusted publishing
    permissions:
      id-token: write
      contents: read

    steps:
      - uses: actions/checkout@v4

      - uses: actions/setup-python@v5
        with:
          python-version: "3.11"
          cache: pip

      - name: Install build tools
        run: pip install build twine

      - name: Build distribution
        run: python -m build

      - name: Check distribution
        run: twine check dist/*

      - name: Publish to PyPI (OIDC trusted publishing — no token required)
        uses: pypa/gh-action-pypi-publish@release/v1
        # No password/token needed — PyPI trusts GitHub's OIDC identity.
        # Configure at pypi.org > Your project > Publishing > Add Trusted Publisher.
'''

(gha_dir / "release.yml").write_text(RELEASE_YML.lstrip())
print("✅ Written .github/workflows/release.yml")

# ─── Print CI flow ────────────────────────────────────────────────────────────
print("""
CI/CD pipeline summary:

  Every push to main or PR:
  lint → test (3.10/3.11/3.12) → smoke (main only) → docs build

  Push tag v1.0.0:
  build → twine check → publish to PyPI (OIDC — zero secrets)

  Deploy docs:
  mkdocs gh-deploy --force   (runs manually or add to release.yml)
""")

# 💡 EXPERIMENT: Add a nightly eval job (separate eval.yml) that runs your
# golden-set evaluation from L49 and posts the score to a Slack channel.

---
## 10 Pitfalls When Shipping Open-Source AI Packages

| # | Pitfall | Consequence | Fix |
|---|---------|-------------|-----|
| 1 | **Releasing without a smoke test** | v1.0.0 raises `KeyError` for the first user who tries it | Smoke test is a mandatory gate in `release.sh` |
| 2 | **Hardcoded model string in package code** | Anthropic deprecates `claude-sonnet-4-5` in 6 months; all users break | Expose `model` as a config param with a sensible default |
| 3 | **API key in test file** | `git log` exposes it forever even after deletion | Use `pytest.mark.skipif` + `os.getenv` guard; never hardcode |
| 4 | **Uploading to real PyPI before TestPyPI** | Bad metadata, broken wheel, wrong version — can't undo a release | Always TestPyPI first: `twine upload --repository testpypi dist/*` |
| 5 | **Missing `py.typed` marker** | Mypy users get `error: Skipping analyzing "auto_researcher"` | Add empty `auto_researcher/py.typed` file; include it in package data |
| 6 | **Pinned transitive deps in `dependencies`** | Conflicts with users' existing environments ("dependency hell") | Only pin direct deps; use `>=` lower bounds, not `==` |
| 7 | **`[Unreleased]` CHANGELOG never rotated** | Release notes on GitHub show empty | Automate it in `release.sh` with `sed` |
| 8 | **OIDC Trusted Publisher not configured** | `release.yml` workflow fails on first tag push | Configure on pypi.org before the first release |
| 9 | **No `__version__` in `__init__.py`** | Users can't check `auto_researcher.__version__` to debug | Always expose `__version__ = importlib.metadata.version('auto-researcher')` |
| 10 | **Smoke test hits real API in every CI run** | $50/month in smoke-test costs | Gate smoke behind `@pytest.mark.integration` + run only on `main` push |


In [ ]:
# ── PITFALL DEMO: #5 — Missing py.typed marker ────────────────────────────────
# This is the most common silent pitfall for library authors.

print("=" * 60)
print("PITFALL #5 DEMO: Missing py.typed marker")
print("=" * 60)

# Without py.typed, mypy users see:
BAD_PYPROJECT_SNIPPET = '''
[tool.setuptools.packages.find]
where = ["."]
# NO package-data config — py.typed not shipped
'''

GOOD_PYPROJECT_SNIPPET = '''
[tool.setuptools.packages.find]
where = ["."]

[tool.setuptools.package-data]
auto_researcher = ["py.typed"]   # ← tells mypy this is a typed package
'''

print("BAD (py.typed missing):")
print(BAD_PYPROJECT_SNIPPET)
print("What mypy users see:")
print('  error: Skipping analyzing "auto_researcher": module is installed, but missing library stubs')
print('  note: See https://mypy.readthedocs.io/en/stable/running_mypy.html#missing-imports')

print("\nFIX: Add py.typed + declare it in package-data:")
print(GOOD_PYPROJECT_SNIPPET)

# Apply the fix
py_typed = pkg_dir / "py.typed"
py_typed.touch()  # empty marker file, PEP 561
print(f"✅ Created {py_typed}  (empty marker file, PEP 561)")

# Also fix __version__
INIT_PY = '''
"""auto-researcher: production-grade AI research assistant."""
from importlib.metadata import version, PackageNotFoundError

try:
    __version__: str = version("auto-researcher")
except PackageNotFoundError:          # running from source without install
    __version__ = "0.0.0.dev0"

__all__ = ["__version__"]
'''
(pkg_dir / "__init__.py").write_text(INIT_PY.lstrip())
print("✅ Updated __init__.py with importlib.metadata version lookup")

print()
print("Rule: Every typed library package must include an empty py.typed file.")
print("Reference: PEP 561 — Distributing and Packaging Type Information")

---
## Final Verification — What We Shipped


In [ ]:
# ── FINAL VERIFICATION ────────────────────────────────────────────────────────
import os

checklist = [
    ("Integration smoke test",            ROOT / "auto_researcher" / "__init__.py"),  # proxy
    ("release.sh",                         ROOT / "release.sh"),
    ("pyproject.toml",                     ROOT / "pyproject.toml"),
    ("README.md with badges",             ROOT / "README.md"),
    (".github/workflows/ci.yml",           ROOT / ".github/workflows/ci.yml"),
    (".github/workflows/release.yml",      ROOT / ".github/workflows/release.yml"),
    ("auto_researcher/py.typed (PEP 561)", ROOT / "auto_researcher/py.typed"),
    ("auto_researcher/__init__.py",        ROOT / "auto_researcher/__init__.py"),
    ("dist/ (built wheel + sdist)",        ROOT / "dist"),
]

print("Lesson 55 — Ship It! Verification")
print("=" * 50)
all_ok = True
for label, path in checklist:
    exists = path.exists()
    icon = "✅" if exists else "❌"
    print(f"  {icon}  {label}")
    if not exists:
        all_ok = False

print()
if all_ok:
    print("🟢 All artifacts present — ready to ship.")
else:
    print("🔴 Some artifacts missing — re-run the cells above.")

print()
print("Phase 5 complete: L51 Architecture → L52 Retrieval → L53 Production")
print("                  → L54 DevEx → L55 Ship It ✅")

---
## 🏆 Full Curriculum Retrospective

You've completed the entire curriculum. Here's what you built, track by track:

| Phase | Track | Lessons | What You Built |
|-------|-------|---------|----------------|
| Phase 1 | LLM Fundamentals | L01–L10 | Prompt engineering, tool use, RAG, streaming, cost control |
| Phase 2 | AI Agents | L11–L25 | ReAct loop, tool agents, memory, multi-agent orchestration |
| Phase 3 | Production Reliability | L26–L36 | Circuit breakers, fallback chains, durable pipelines, observability |
| Phase 4 | Track 1: Reliability | L31–L36 | Full reliability spine: retry/fallback/budget/eval |
| Phase 4 | Track 2: Agent Swarms | L36–L41 | A2A protocol, swarm orchestration, task routing |
| Phase 4 | Track 3: Self-Hosted LLMs | L37–L41 | vLLM serving, QLoRA, DPO, distillation, model merging |
| Phase 4 | Track 4: Multimodal | L42–L45 | Voice (Whisper→TTS), image gen tools, Document AI |
| Phase 4 | Track 5: Observability | L46–L50 | Durable execution, GPU autoscaling, OTel tracing, eval at scale |
| **Phase 5** | **OSS Ship It** | **L51–L55** | **Full OSS package: arch → retrieval → production → devex → PyPI** |

---

### What makes you an AI engineer now

You can reason about and implement **every layer** of a production AI system:

```
┌─────────────────────────────────────────────────────────┐
│                  User-facing product                    │
├─────────────────────────────────────────────────────────┤
│  Agent Layer    │  Tool use · Swarms · A2A protocol     │
├─────────────────────────────────────────────────────────┤
│  Inference      │  Anthropic API · vLLM · QLoRA · DPO  │
├─────────────────────────────────────────────────────────┤
│  Retrieval      │  Hybrid search · Re-ranking · HyDE    │
├─────────────────────────────────────────────────────────┤
│  Reliability    │  Circuit breakers · Durable exec ·    │
│                 │  Budget guards · Retry policies        │
├─────────────────────────────────────────────────────────┤
│  Observability  │  OTel traces · Prometheus · Eval at   │
│                 │  scale · Regression detection          │
├─────────────────────────────────────────────────────────┤
│  DevEx & Ops    │  Docker · FastAPI · CI/CD · PyPI      │
└─────────────────────────────────────────────────────────┘
```

---

## What to Build Next — Proving Your Worth

The curriculum is complete. Now you need **GitHub evidence**. Here are 3 project ideas that combine everything you've learned:

### 🔬 Project A: `paper-distiller` (recommended first project)
- Input: arXiv paper URL
- Pipeline: PDF extraction → section chunking → hybrid RAG → structured summary (contributions + benchmarks + limitations)
- Output: JSON + Markdown summary + voice narration
- Why: Directly demonstrates Tracks 2, 3, 4, 5 all wired together
- OSS angle: arXiv researchers would actually use this

### 🤖 Project B: `agent-bench` (evaluations-focused)
- A benchmark harness for comparing AI agents across tasks
- Implements: task registry + agent protocol + eval metrics + leaderboard
- Why: Evaluation is underserved — this fills a real gap
- OSS angle: Connect to HuggingFace datasets

### 🏗️ Project C: `auto-researcher` v1.0.0 (what you built)
- Ship what you already have
- Add: a hosted demo (Hugging Face Spaces or Modal), real docs on GitHub Pages, a blog post
- Why: Done > perfect. A published package beats an unpublished "almost done" project

---

**The one rule for open-source credibility:**
> Ship something that solves a real problem, has tests, and has a README that explains why it exists.
> Everything else is polish.

---
## Homework (5 Challenges)

1. **Run the smoke test** against your real `auto_researcher_v2` pipeline (not the inline version here). Fix anything that breaks.

2. **Publish to TestPyPI.** Build the wheel, run `twine check dist/*`, upload to `test.pypi.org`. Install it with `pip install --index-url https://test.pypi.org/simple/ auto-researcher` and verify `import auto_researcher; print(auto_researcher.__version__)` works.

3. **Set up OIDC Trusted Publishing.** On pypi.org (you'll need an account), create the project and add a Trusted Publisher pointing to your GitHub repo + `release.yml`. Then push a `v0.0.1` tag and watch the Actions workflow publish to PyPI automatically.

4. **Add a Codecov badge that's green.** Connect your repo to codecov.io (free for open source), add the `CODECOV_TOKEN` secret to GitHub, and make the coverage badge show ≥ 80% by writing the tests you need.

5. **Write a blog post** (500–800 words) describing what `auto-researcher` does, how you built it, and what you learned. Post it anywhere (dev.to, Medium, LinkedIn, your own site). Link it from the README. This is your public proof of work.

---

## Summary

| Concept | What you learned |
|---------|------------------|
| Integration smoke test | End-to-end gate before every release; structural assertions, not semantic |
| `release.sh` | Automates: clean-tree check → tests → smoke → version bump → CHANGELOG → tag → gh release |
| `python -m build` | Produces `.whl` (wheel) and `.tar.gz` (sdist) from `pyproject.toml` |
| `twine check` | Validates metadata before upload — catches 80% of upload failures |
| OIDC Trusted Publishing | PyPI trusts GitHub Actions identity — no long-lived tokens to manage |
| README badges | Shields.io badges auto-update from PyPI + GitHub Actions data |
| `py.typed` (PEP 561) | Empty marker file that enables mypy checking for your library |
| Two-workflow CI/CD | `ci.yml` (every push) vs `release.yml` (tag push only) separation of concerns |

---

**Phase 5 COMPLETE. Curriculum COMPLETE. 🎓**

You started with zero AI engineering knowledge. You can now:
- Build, evaluate, and deploy production AI agents
- Fine-tune and serve your own models
- Instrument pipelines with real observability
- Ship open-source Python packages that the community can `pip install`

The rest is building. Go ship something real.